# ScaleRAG – Multimodal RAG Retrieval Pipeline (v1)

This notebook builds the first version of our **multimodal Retrieval-Augmented Generation (RAG)** pipeline using the preprocessed chunks from the data preparation stage. We load the merged text and image chunks and compute dense embeddings for retrieval.

A **SentenceTransformer model** (`all-MiniLM-L6-v2`) produces **384-dimensional embeddings** for text, while **OpenAI’s CLIP model** (`ViT-B/32`) produces **512-dimensional embeddings** for images.  
When a chunk has both an image and caption, we concatenate the normalized caption and image vectors to form an **896-dimensional embedding**.  
All embeddings are normalized and saved, and **FAISS indexes** are built for fast similarity search.  

---

## Pipeline Overview

1. **Model Setup:**  
   Initialize the SentenceTransformer for text and the CLIP model (with processor) for images, and configure the compute device (GPU/CPU).

2. **Load Chunks:**  
   Read all preprocessed RAG chunk files from `data/rag_chunks/*.json` into a single list.

3. **Separate by Type:**  
   Split the merged chunks into text chunks (*paragraph, text, equation*) and image chunks (*figures, tables*).

4. **Embed Text Chunks:**  
   Batch-encode all text contents using the text model, normalize the 384-D vectors, and attach them to chunk records.

5. **Embed Image Chunks:**  
   For each figure/table, use CLIP to extract a 512-D image feature, normalize it, and if a caption exists, encode it with the text model and concatenate (resulting in 896-D). Attach the combined embeddings.

6. **Save Embeddings:**  
   Store the list of embedded chunks to disk in both JSON and Pickle formats for reuse.

7. **Build FAISS Indexes:**  
   Collect text and image vectors into NumPy arrays, create **FAISS IndexFlatIP** indexes (inner-product for cosine similarity), add the vectors, and save the indexes to disk.

8. **Retrieval Demo:**  
   Define query functions to embed user queries and retrieve the top-k relevant text or image chunks via the FAISS indexes.

---

## Output Directory Summary

- `data/RAG/embeddings/all_papers.embeddings.json` – JSON file of all chunk records with embeddings  
- `data/RAG/embeddings/all_papers.embeddings.pkl` – Pickle file of the same embedding data  
- `data/RAG/indexes/text.index.faiss` – FAISS index for text embeddings (384-D)  
- `data/RAG/indexes/image.index.faiss` – FAISS index for image embeddings (896-D)



## Model and Tokenizer Setup (CLIP & SentenceTransformer)

- **SentenceTransformer:**  
  Uses `all-MiniLM-L6-v2` for 384-dimensional text embeddings.

- **CLIP Model:**  
  Loads `openai/clip-vit-base-patch32` and its processor for 512-dimensional image feature extraction.

- **Device Configuration:**  
  Sets `device = cuda` if available, otherwise `cpu`. The CLIP model is moved to this device.  
  The text model remains on CPU in this notebook (but can also be moved to GPU if desired).

In [1]:
import torch
from PIL import Image
# Text model (SentenceTransformer)
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Vision model (CLIP)
from transformers import CLIPProcessor, CLIPModel
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

# Ensure models are on CPU or GPU as available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model = clip_model.to(device)
text_model = text_model


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Loading Preprocessed RAG Chunks

In this step, we load all preprocessed chunk files created during the data preparation phase.

1. **Load JSON Files:**  
   Read all chunk files from `data/rag_chunks/*.json`.  
   Each file contains a list of chunk records with fields such as `id`, `type`, `content`, and `metadata`.

2. **Merge Lists:**  
   Combine the contents of all files into a single list named `merged_chunks`.  
   Print the total number of chunks and display one example record to confirm the format.


In [42]:
import json
import glob

merged_chunks = []

for file_path in glob.glob("data/rag_chunks_v2/*.json"):
    with open(file_path, "r") as f:
        file_chunks = json.load(f)
        merged_chunks.extend(file_chunks)

print("Total merged chunks:", len(merged_chunks))
print("First example chunk:")
print(merged_chunks[0])


Total merged chunks: 2330
First example chunk:
{'id': '2406.03243_paragraph_4', 'type': 'paragraph', 'content': 'Inference serving of LLMs plays a key role in LLMpowered services, becoming a critical workload in datacenters. Such services are typically backed by multiple instances of the LLM deployed on a GPU cluster. The system involves a scheduler and an inference engine , where a request is first dispatched by the scheduler to a model serving instance, then gets executed by the inference engine inside. The requests are typically batched for execution on each instance to increase throughput and cost efficiency.', 'metadata': {'page': 1, 'section': 'Llumnix: Dynamic Scheduling for Large Language Model Serving', 'bbox': [317.88, 432.33, 559.747, 536.523]}, 'source_ids': ['2406.03243_paragraph_4']}


## Separating Text and Image Chunks

In this step, we categorize the merged RAG chunks based on their content type.

1. **Filter by Type:**  
   Create two separate lists from `merged_chunks`:  
   - `text_chunks` → where `type` is `"paragraph"`, `"text"`, or `"equation"`  
   - `image_chunks` → where `type` is `"figure"` or `"table"`

2. **Sanity Check:**  
   Print the counts of text vs. image chunks to verify the split.  
   This ensures that each chunk type will later be embedded using the correct model (text or image encoder).


In [3]:
import numpy as np
from torch.nn.functional import normalize

embedded_chunks = []  # final list of chunk dicts with embeddings

text_chunks = [ch for ch in merged_chunks if ch['type'] in ['paragraph', 'text', 'equation']]
image_chunks = [ch for ch in merged_chunks if ch['type'] in ['figure', 'table']]

print(f"📄 Text chunks: {len(text_chunks)} | 🖼️ Image chunks: {len(image_chunks)}")

📄 Text chunks: 1960 | 🖼️ Image chunks: 370


In [4]:
len(text_chunks), len(image_chunks)

(1960, 370)

In [5]:
text_chunks[0]

{'id': '2406.03243_paragraph_4',
 'type': 'paragraph',
 'content': 'Inference serving of LLMs plays a key role in LLMpowered services, becoming a critical workload in datacenters. Such services are typically backed by multiple instances of the LLM deployed on a GPU cluster. The system involves a scheduler and an inference engine , where a request is first dispatched by the scheduler to a model serving instance, then gets executed by the inference engine inside. The requests are typically batched for execution on each instance to increase throughput and cost efficiency.',
 'metadata': {'page': 1,
  'section': 'Llumnix: Dynamic Scheduling for Large Language Model Serving',
  'bbox': [317.88, 432.33, 559.747, 536.523]},
 'source_ids': ['2406.03243_paragraph_4']}

## Embedding Text Chunks

In this step, we compute dense embeddings for all textual RAG chunks using the SentenceTransformer model.

1. **Batch Encoding:**  
   Extract all text contents from `text_chunks` and encode them in batches using:  
   ```python
   text_model.encode(texts, batch_size=32)
   ```  
   The output is a NumPy array of shape *(num_text_chunks, 384)*, where each row represents a 384-dimensional embedding vector.

2. **Normalization:**  
   Normalize each 384-D vector by dividing it by its L2 norm so that all vectors lie on the unit sphere.  
   This ensures that cosine similarity can be computed directly using the inner product.

3. **Attach to Chunks:**  
   For each text chunk, add the normalized embedding (converted to a Python list) under the key `"embedding"`.  
   The updated record structure becomes:  
    ```python
   {
       "id": ...,
       "type": ...,
       "content": ...,
       "metadata": ...,
       "embedding": [...]
   }
   ```

In [6]:
# Batch compute text embeddings for all text chunks to speed up
texts = [ch['content'] for ch in text_chunks]
text_embeds = text_model.encode(texts, batch_size=32, show_progress_bar=True)
# text_embeds will be a numpy array of shape (len(text_chunks), 384) in this case

# Normalize text embeddings
text_embeds = text_embeds / np.linalg.norm(text_embeds, axis=1, keepdims=True)

# Assign back the text embeddings to their chunks
for ch, vec in zip(text_chunks, text_embeds):
    vec_list = vec.tolist()
    ch_emb = vec_list  # 384-dim text embedding
    # we handle that in the image loop instead.
    embedded_chunks.append({
        "id": ch["id"],
        "type": ch["type"],
        "content": ch["content"],
        "metadata": ch.get("metadata", {}).copy(),
        "embedding": ch_emb
    })

print(f"Text embeddings computed for {len(text_chunks)} chunks.")

Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Text embeddings computed for 1960 chunks.


## Embedding Image Chunks (Figures/Tables)

In this step, we generate embeddings for all image-based RAG chunks such as figures and tables.

1. **Open Image:**  
   For each chunk in `image_chunks`, open the associated image file using PIL:  
   ```python
   image = Image.open(path).convert("RGB")
   ```  
   Skip any chunks where the image cannot be loaded or the file path is missing.

2. **CLIP Encoding:**  
   Preprocess each image using the CLIP processor and extract image features.
   This produces a **512-dimensional image feature vector**.

3. **Normalize Image Vector:**  
   Divide the 512-D image vector by its L2 norm to ensure unit length.

4. **Caption Encoding (if present):**  
   If the chunk’s content contains a caption text, encode it using the text model to obtain a **384-D vector**, then normalize it.

5. **Combine Features:**  
   - If the two vectors have matching shapes (rare case), average them.  
   - Otherwise, concatenate the 384-D caption vector and 512-D image vector to form an **896-D combined embedding**.

6. **Attach to Chunks:**  
   Add the combined 896-D embedding (converted to a Python list) under the key `"embedding"`.  
   The final record structure becomes:  
   ```python
   {
       "id": ...,
       "type": ...,
       "content": ...,
       "metadata": ...,
       "embedding": [...]
   }
   ```


In [7]:
# Now handle image-containing chunks (figures, tables)
for ch in image_chunks:
    img_path = ch.get("metadata", {}).get("image_path") or ch.get("image_path")
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Warning: could not open image at {img_path}: {e}")
        continue  # skip if image not available
    # Preprocess image for CLIP
    inputs = clip_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    # Get CLIP image feature (512-dim)
    with torch.no_grad():
        image_feat = clip_model.get_image_features(pixel_values=pixel_values)
    image_vec = image_feat.cpu().numpy().flatten()
    # Normalize image embedding
    image_vec = image_vec / np.linalg.norm(image_vec)

    # Check if there's associated text (e.g., a caption in content)
    combined_vec = image_vec
    if ch.get("content"):
        caption = str(ch["content"])
        # Use the same text model for caption
        caption_emb = text_model.encode([caption])[0]
        caption_emb = caption_emb / np.linalg.norm(caption_emb)
        # Concatenate or average with image_vec:
        if caption_emb.shape[0] == image_vec.shape[0]:
            # If by chance using CLIP text encoder for caption, shapes align (512 each)
            combined_vec = (image_vec + caption_emb) / 2.0  # average
        else:
            # Different dimensions (e.g., 384 vs 512), concatenate
            combined_vec = np.concatenate([caption_emb, image_vec])
            # (combined_vec is now 896-dim in this scenario)
            # We could optionally reduce dimension or keep as is for indexing.
    else:
        # No caption text, combined_vec stays as image_vec
        pass

    embedded_chunks.append({
        "id": ch["id"],
        "type": ch["type"],
        "content": ch.get("content", ""),  # might be caption or empty
        "metadata": ch.get("metadata", {}).copy(),
        "embedding": combined_vec.tolist()
    })

print(f"Computed embeddings for {len(embedded_chunks)} chunks.")
# Show an example of a text chunk and an image chunk embedding (truncated for display)
for ex in embedded_chunks[:2]:
    print(f"{ex['type']} chunk '{ex['id']}' -> embedding length {len(ex['embedding'])}, sample: {ex['embedding'][:5]}")

Computed embeddings for 2330 chunks.
paragraph chunk '2406.03243_paragraph_4' -> embedding length 384, sample: [-0.05061816796660423, -0.051298320293426514, -0.03766610100865364, 0.007535737007856369, 0.0434429831802845]
paragraph chunk '2406.03243_paragraph_8' -> embedding length 384, sample: [-0.06758330017328262, 0.048984672874212265, 0.0064912885427474976, 0.07365099340677261, -0.04049195721745491]


## Saving Embeddings to Disk

In this step, we save the computed embeddings for future use.

1. **Create Directory:**  
   Ensure the folder `data/RAG/embeddings` exists.

2. **Save JSON:**  
   Dump the `embedded_chunks` list to `all_papers.embeddings.json`.

3. **Save Pickle:**  
   Write the same list to `all_papers.embeddings.pkl` using `pickle.dump`.

4. **Verification:**  
   Print confirmation messages after saving to confirm that both files were successfully created.


In [8]:
import os, json, pickle

# Create the folder hierarchy
base_dir = "data/RAG/Version_V2"
emb_dir = os.path.join(base_dir, "embeddings")
os.makedirs(emb_dir, exist_ok=True)

print(f" +++ Folder created at: {emb_dir}")

# Save embeddings
json_path = os.path.join(emb_dir, "all_papers.embeddings.json")
pkl_path = os.path.join(emb_dir, "all_papers.embeddings.pkl")

# Save JSON
with open(json_path, "w") as f:
    json.dump(embedded_chunks, f)
print(f" +++ JSON embeddings saved to {json_path}")

# Save Pickle
with open(pkl_path, "wb") as f:
    pickle.dump(embedded_chunks, f)
print(f" +++ Pickle embeddings saved to {pkl_path}")


 +++ Folder created at: data/RAG/Version_V2/embeddings
 +++ JSON embeddings saved to data/RAG/Version_V2/embeddings/all_papers.embeddings.json
 +++ Pickle embeddings saved to data/RAG/Version_V2/embeddings/all_papers.embeddings.pkl


## Loading and Verifying Saved Embeddings

In this step, we perform a quick sanity check to ensure that the saved embedding files are valid and correctly structured.

1. **Load Pickle:**  
   Reload the embeddings list from `all_papers.embeddings.pkl`.

2. **Inspect Structure:**  
   Print the total number of chunks, view example keys from the first record (`id`, `type`, `content`, etc.), and check the embedding dimensions.

3. **Verification:**  
   Confirm that text chunks have **384-D embeddings** and image chunks have **896-D embeddings**.


In [1]:
import pickle

with open("../data2/RAG/Version_V2/embeddings/all_papers.embeddings.pkl", "rb") as f:
    embedded_chunks = pickle.load(f)

print("Loaded", len(embedded_chunks), "chunks.")
print("Example record keys:", embedded_chunks[0].keys())
print("Example metadata:", embedded_chunks[0]["metadata"])
print("Embedding dim:", len(embedded_chunks[0]["embedding"]))

Loaded 75431 chunks.
Example record keys: dict_keys(['id', 'type', 'content', 'metadata', 'embedding'])
Example metadata: {'page': 1, 'section': 'KEYWORDS', 'bbox': [53.484, 264.288, 295.032, 287.638]}
Embedding dim: 384


In [3]:
import numpy as np

text_dims = [len(ch["embedding"]) for ch in embedded_chunks if ch["type"] in ["text","paragraph","equation"]]
image_dims = [len(ch["embedding"]) for ch in embedded_chunks if ch["type"] in ["figure","table"]]

print(f"Text embeddings: mean {np.mean(text_dims):.0f} ± {np.std(text_dims):.1f}")
print(f"Image embeddings: mean {np.mean(image_dims):.0f} ± {np.std(image_dims):.1f}")

Text embeddings: mean 384 ± 0.0
Image embeddings: mean 896 ± 0.0


## Splitting Embeddings for Indexing

In this step, we separate the embeddings by type to prepare for FAISS indexing.

1. **Filter by Dimension:**  
   Iterate over all loaded `embedded_chunks`.  
   - If the type is a text type and the embedding length is **384**, append it to `text_records` and collect its vector.  
   - If the type is an image/table and the embedding length is **896**, append it to `image_records` and collect its vector.  
   - Ignore any records with unexpected embedding dimensions.

2. **Stack Vectors:**  
   Convert the collected vectors into NumPy arrays:  
   - `text_vectors` → shape `(N_text, 384)`  
   - `image_vectors` → shape `(N_image, 896)`

3. **Print Shapes:**  
   Display the number of records and array shapes to verify the data integrity before indexing.


## Building and Saving FAISS Indexes

1. **Create Indexes:**  
   Initialize FAISS indexes using **inner product (IP)** similarity, which corresponds to cosine similarity when embeddings are normalized.  
   - `index_text = IndexFlatIP(384)` for text embeddings  
   - `index_image = IndexFlatIP(896)` for image embeddings

2. **Add Vectors:**  
   Add `text_vectors` to `index_text` and `image_vectors` to `index_image`.  
   The final index sizes should match the number of added vectors.

3. **Save to Disk:**  
   Ensure the folder `data/RAG/indexes` exists.  
   Save the indexes as `text.index.faiss` and `image.index.faiss`, and print confirmation messages to verify successful writes.


In [4]:
import numpy
import faiss
print("NumPy version:", numpy.__version__)
print("FAISS version:", faiss.__version__)

NumPy version: 1.26.4
FAISS version: 1.7.2


In [5]:
import numpy as np
import faiss

# --- Split into text chunks (384-D) and figure/table chunks (896-D)

text_records = []
text_vectors = []

image_records = []
image_vectors = []

for ch in embedded_chunks:
    vec = np.array(ch["embedding"], dtype="float32")
    dim = vec.shape[0]

    if ch["type"] in ["text", "paragraph", "equation", "table_caption", "figure_caption"]:
        # safety check: only keep consistent dim=384
        if dim == 384:
            text_records.append(ch)
            text_vectors.append(vec)
    elif ch["type"] in ["figure", "table"]:
        # safety check: only keep consistent dim=896 (caption+image concat case)
        if dim == 896:
            image_records.append(ch)
            image_vectors.append(vec)
    else:
        # ignore other types for now or print to debug
        pass

text_vectors = np.stack(text_vectors, axis=0) if len(text_vectors) > 0 else np.zeros((0,384), dtype="float32")
image_vectors = np.stack(image_vectors, axis=0) if len(image_vectors) > 0 else np.zeros((0,896), dtype="float32")

print("Text records:", len(text_records), "| text_vectors shape:", text_vectors.shape)
print("Image records:", len(image_records), "| image_vectors shape:", image_vectors.shape)

# --- Build FAISS indexes

dim_text = text_vectors.shape[1]
dim_image = image_vectors.shape[1]

index_text = faiss.IndexFlatIP(dim_text)    # cosine sim if vectors are normalized
index_image = faiss.IndexFlatIP(dim_image)  # cosine sim if vectors are normalized

index_text.add(text_vectors)
index_image.add(image_vectors)

print("FAISS index_text size:", index_text.ntotal)
print("FAISS index_image size:", index_image.ntotal)

# Save indexes to disk so you don't have to rebuild next time
import os
os.makedirs("../data2/RAG/Version_V2/indexes", exist_ok=True)

faiss.write_index(index_text, "../data2/RAG/Version_V2/indexes/text.index.faiss")
faiss.write_index(index_image, "../data2/RAG/Version_V2/indexes/image.index.faiss")

print(" +++ Saved FAISS indexes.")


Text records: 1960 | text_vectors shape: (1960, 384)
Image records: 370 | image_vectors shape: (370, 896)
FAISS index_text size: 1960
FAISS index_image size: 370
 +++ Saved FAISS indexes.


## Defining Query and Retrieval Functions

In this section, we define helper functions that enable natural language queries over both text and image embeddings.

1. **Text Query Embedding:**  
   Define `embed_query_text(query)` to encode an input string using the text model and normalize it into a **384-D vector**.  
   This vector will be used for querying the text FAISS index.

2. **Image-side Query Embedding:**  
   Define `embed_query_clip_text(query)` to compute CLIP’s text embedding (**512-D**) and normalize it.  
   Then implement `build_query_for_image_index(query)` to concatenate:  
   - The normalized **text-model embedding (384-D)**  
   - The normalized **CLIP text embedding (512-D)**  
   forming a combined **896-D query vector** for the image index.

3. **Retrieve Functions:**  
   - `retrieve_text(query, k)`: Embeds the query, searches `index_text`, and returns the top-k text chunks with their scores and metadata.  
   - `retrieve_image(query, k)`: Builds the 896-D query vector, searches `index_image`, and returns the top-k figure or table chunks.

These retrieval functions enable semantic search over both textual and visual chunks using natural language queries.


In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model = clip_model.to(device)

def embed_query_text(query: str):
    """
    Embed the query using the SAME SentenceTransformer you used for text chunks.
    Returns a normalized 384-D np.array(float32) of shape (1, 384).
    """
    q_vec = text_model.encode([query])  # shape (1,384) as float32/float64
    q_vec = q_vec / np.linalg.norm(q_vec, axis=1, keepdims=True)
    return q_vec.astype("float32")

def embed_query_clip_text(query: str):
    """
    Use CLIP's text tower to embed the query in CLIP space (512-D).
    This lets us search the 'image side' because CLIP aligns text<->image.
    We'll L2-normalize it.
    """
    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        # get_text_features gives CLIP's text embedding
        text_feat = clip_model.get_text_features(**inputs)
        # normalize
        text_feat = text_feat / text_feat.norm(p=2, dim=-1, keepdim=True)

    vec = text_feat.squeeze(0).detach().cpu().numpy()  # (512,)
    return vec.astype("float32")

def build_query_for_image_index(query: str):
    """
    Build the 896-D query vector that matches how figure/table embeddings were built:
    [384-D text_model embedding | 512-D CLIP text embedding]
    Both halves are normalized separately first.
    Returns shape (1, 896).
    """
    q_caption_384 = embed_query_text(query)[0]        # (384,)
    q_cliptext_512 = embed_query_clip_text(query)     # (512,)

    q_concat_896 = np.concatenate([q_caption_384, q_cliptext_512], axis=0)
    # final normalize whole vector so cosine/IP works nicely
    q_concat_896 = q_concat_896 / np.linalg.norm(q_concat_896, keepdims=True)

    return q_concat_896.astype("float32")[None, :]    # shape (1,896)


In [7]:
def retrieve_text(query, k=5):
    """
    Search the text FAISS index.
    Returns the top-k text chunks with scores and metadata.
    """
    q_vec = embed_query_text(query)  # (1,384)
    scores, idxs = index_text.search(q_vec, k)  # cosine/IP scores

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0])):
        ch = text_records[idx]
        results.append({
            "rank": rank,
            "score": float(score),
            "id": ch["id"],
            "type": ch["type"],
            "page": ch["metadata"].get("page"),
            "section": ch["metadata"].get("section"),
            "content_preview": ch["content"][:400],
        })
    return results

def retrieve_image(query, k=5):
    """
    Search the figure/table FAISS index using the 896-D query vector.
    Returns top-k visual chunks (figures/tables).
    """
    q_vec = build_query_for_image_index(query)  # (1,896)
    scores, idxs = index_image.search(q_vec, k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0])):
        ch = image_records[idx]
        results.append({
            "rank": rank,
            "score": float(score),
            "id": ch["id"],
            "type": ch["type"],
            "page": ch["metadata"].get("page"),
            "section": ch["metadata"].get("section"),
            "image_path": ch["metadata"].get("image_path"),
            "content_preview": ch.get("content", "")[:400],  # caption or ""
        })
    return results

def retrieve_multimodal(query, k_text=5, k_image=5):
    """
    Convenience wrapper: gets both text and image/table hits.
    """
    text_hits = retrieve_text(query, k=k_text)
    image_hits = retrieve_image(query, k=k_image)
    return text_hits, image_hits


## Retrieval Demo – Multimodal Query

In this step, we test the multimodal retrieval pipeline using a natural language question.  
The query is embedded for both text and image spaces, and the top-k most relevant chunks are retrieved from the FAISS indexes.

The output below displays:
- **Text hits:** top-5 paragraphs or sections ranked by similarity score.  
- **Image/Figure/Table hits:** top-5 visual elements with captions and page references.

This confirms that the RAG system can retrieve semantically relevant text and images for complex research-style queries.


In [8]:
question = "What methods exist to accelerate token generation during inference without retraining the language model in Medusa paper?"
text_hits, image_hits = retrieve_multimodal(question, k_text=5, k_image=5)

print("=== TEXT HITS ===")
for h in text_hits:
    print(f"[rank {h['rank']} score {h['score']:.3f}] page {h['page']} section {h['section']}")
    print(h["content_preview"])
    print("---")

print("\n=== IMAGE / FIGURE / TABLE HITS ===")
for h in image_hits:
    print(f"[rank {h['rank']} score {h['score']:.3f}] {h['type']} page {h['page']} section {h['section']}")
    print("image_path:", h["image_path"])
    print("caption/summary:", h["content_preview"])
    print("---")


=== TEXT HITS ===
[rank 0 score 0.647] page 9 section Discussion
In conclusion, MEDUSA enhances LLM inference speed by 2.3-2.8 times by equipping models with additional predictive decoding heads, allowing for generating multiple tokens simultaneously and bypassing the sequential decoding limitation. Key advantages of MEDUSA include its simplicity, parameter efficiency, and ease of integration into existing systems. MEDUSA avoids the need for specialized draft m
---
[rank 1 score 0.632] page 16 section References
Borgeaud, A. Mensch, J. Hoffmann, T. Cai, E. Rutherford, K. Millican, G. van den Driessche, J.-B. Lespiau, B. Damoc, A. Clark, D. de Las Casas, A. Guy, J. Menick, R. Ring, T. Hennigan, S. Huang, L. Maggiore, C. Jones, A. Cassirer, A. Brock, M. Paganini, G. Irving, O. Vinyals, S. Osindero, K. Simonyan, J. W. Rae, E. Elsen, and L. Sifre. Improving language models by retrieving from trillions of toke
---
[rank 2 score 0.607] page 3 section RELATED WORK
Token Dropping and KV Cache 

## Building Combined Context for RAG Generation

Here we define the `build_context()` function, which merges top-ranked text and image retrievals into a unified context block.  
For each query, it gathers the top-k text and image hits, formats their content, and concatenates them into a single context string.  

This context will later be passed to the generation model to enable multimodal, context-aware responses.


In [17]:
def build_context(query, k_text=5, k_image=3):
    text_hits, image_hits = retrieve_multimodal(query, k_text, k_image)

    context_parts = []
    for h in text_hits:
        context_parts.append(f"[Text: page {h['page']} - {h['section']}]\n{h['content_preview']}")

    for h in image_hits:
        context_parts.append(
            f"[{h['type'].upper()}: page {h['page']} - {h['section']}]\n"
            f"Caption: {h['content_preview']}\nImage path: {h['image_path']}"
        )

    context = "\n\n".join(context_parts)
    return context


In [9]:
from pathlib import Path

def build_mm_context(query, k_text=5, k_image=3, max_text_chars=1200):
    """
    Build a multimodal context for vision-language models (LLaVA, Qwen-VL, etc.)
    Returns a *list* of content blocks, not a single string.
    Each block is either:
      {"type": "text", "text": "..."}  or
      {"type": "image", "image": "/path/to/image.png"}
    """
    text_hits, image_hits = retrieve_multimodal(query, k_text=k_text, k_image=k_image)

    content = []

    # 1) add text chunks first, in rank order
    for i, h in enumerate(text_hits, start=1):
        txt = h.get("content_preview") or h.get("content") or ""
        if len(txt) > max_text_chars:
            txt = txt[:max_text_chars] + " ..."
        page = h.get("page", "?")
        section = h.get("section", "?")
        content.append({
            "type": "text",
            "text": f"[TEXT #{i} | page {page} | section {section}]\n{txt}"
        })

    # 2) add figures / tables: caption + actual image
    for i, h in enumerate(image_hits, start=1):
        cap = h.get("content_preview") or h.get("content") or ""
        page = h.get("page", "?")
        section = h.get("section", "?")
        img_path = h.get("image_path")

        # caption first
        content.append({
            "type": "text",
            "text": (
                f"[{h.get('type','image').upper()} #{i} | page {page} | section {section}]\n"
                f"Caption: {cap}"
            )
        })

        # then the actual image, if we have a path
        if img_path and Path(img_path).exists():
            content.append({
                "type": "image",
                "image": img_path
            })
        else:
            # fallback: tell the model there was an image we couldn't load
            content.append({
                "type": "text",
                "text": "(image referenced in paper, but file not found on disk)"
            })

    return content


In [10]:
mm_context = build_mm_context(
    "Explain tree attention in Medusa paper?",
    k_text=5,
    k_image=3
)
mm_context

[{'type': 'text',
  'text': "[TEXT #1 | page 8 | section CONFIGURATION OF TREE ATTENTION]\nThe study of tree attention is conducted on the writing and roleplay categories from the MT-Bench dataset using MEDUSA-2 Vicuna-7B. We target to depict tree attention's motivation and its performance. Fig. 4a compares the acceleration rate of randomly sampled dense tree configurations (Section. 2.1.2, depicted by blue dots) against optimized sparse tree settings (Section. 2.3.3, shown with red sta"},
 {'type': 'text',
  'text': '[TEXT #2 | page 14 | section C. Visualization of optimized tree attention]\nFig. 6 illustrates the structure of a sparsely constructed tree for the MEDUSA-2 Vicuna-7B model. This tree structure extends four levels deep, indicating the engagement of four MEDUSA heads in the computation. The tree is initially formed through a Cartesian product approach and subsequently refined by pruning based on the statistical expectations of the top-k predictions from each MEDUSA head me

## Checking GPU Memory and Selecting Model Configuration

Before loading any large language models, we first check the available GPU memory to decide which model variant and precision can be used efficiently.

This step automatically selects an appropriate **Gemma model (2B or 7B)** and determines whether **4-bit quantization** should be applied based on available VRAM.  
It ensures optimal performance while preventing out-of-memory errors during model loading and inference.


In [11]:
# Get GPU available memory
import torch
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2**30))
print(f"Available GPU memory: {gpu_memory_gb} GB")

Available GPU memory: 15 GB


In [12]:
# Note: the following is Gemma focused, however, there are more and more LLMs of the 2B and 7B size appearing for local use.
if gpu_memory_gb < 5.1:
    print(f"Your available GPU memory is {gpu_memory_gb}GB, you may not have enough memory to run a Gemma LLM locally without quantization.")
elif gpu_memory_gb < 8.1:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in 4-bit precision.")
    use_quantization_config = True
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb < 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.")
    use_quantization_config = False
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb > 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommend model: Gemma 7B in 4-bit or float16 precision.")
    use_quantization_config = False
    model_id = "google/gemma-7b-it"

print(f"use_quantization_config set to: {use_quantization_config}")
print(f"model_id set to: {model_id}")

GPU memory: 15 | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.
use_quantization_config set to: False
model_id set to: google/gemma-2b-it


In [ ]:
# !pip install -U "huggingface_hub[cli]"

In [13]:
from huggingface_hub import login
login()


# Load LLaVA Multimodal for Generation

In [ ]:
# 2) Load LLaVA model + processor
# ------------------------------------------------
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "llava-hf/llava-1.5-7b-hf" 

processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,         # use fp16 for GPU
    low_cpu_mem_usage=True              # reduce CPU memory
).to(device).eval()

In [33]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()


In [ ]:
!nvidia-smi

# Load LLaVA Model in 4-bit mode because of the T4 15GB VRAM limitation.

In [14]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig
)

model_id = "llava-hf/llava-1.5-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # efficient for T4
)

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto"
)
print("LLaVA loaded successfully on T4 in 4-bit mode.")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/mt3846/envTorch124/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2291: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

LLaVA loaded successfully on T4 in 4-bit mode.


# Context Preparation and Prompt Construction

This cell builds the **multimodal input context** for the model by combining retrieved text and figure snippets from the paper.  
It then constructs a structured **prompt** that guides the model to produce an evidence-grounded, citation-aware academic answer.

**Steps performed:**
1. Retrieve top relevant text and figure chunks using `build_mm_context()`.
2. Separate and preprocess text and images for multimodal input.
3. Define a general **task guidance** prompt enforcing academic tone, conciseness, and citation format.
4. Combine everything into a Hugging Face / LLaVA-compatible conversation template.
5. Tokenize and prepare inputs for model inference (with cached tokenizer and EOS/PAD IDs for the next step).


In [49]:
# --- Cell 1: shared guidance + helper ---
from PIL import Image
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

task_guidance = (
    "You are reading an academic paper containing both text and figures. "
    "Use only the provided content to answer the question — do not use outside knowledge. "
    "If some information is missing, state that clearly.\n\n"
    "When answering, follow these principles:\n"
    "1. Be concise and precise — use short paragraphs or bullet points.\n"
    "2. Explicitly cite evidence from the text and figures (e.g., [TEXT #2], [FIGURE #1]).\n"
    "3. Explain concepts in clear academic language, suitable for a research assistant.\n"
    "4. If relevant, describe how the text and figures together support your answer.\n"
    "5. Do NOT include any external URLs or sources outside the paper.\n"
)

def prepare_inputs(query: str, k_text=3, k_image=2):
    """Build mm_context -> conversation -> prompt -> model_inputs for a NEW query."""
    mm_context = build_mm_context(query, k_text=k_text, k_image=k_image)

    text_blocks, images = [], []
    for block in mm_context:
        if block["type"] == "text":
            text_blocks.append(block["text"])
        elif block["type"] == "image":
            images.append(Image.open(block["image"]).convert("RGB"))

    context_text = "\n\n".join(text_blocks)

    conversation = [{
        "role": "user",
        "content": (
            [{"type": "image"} for _ in images] + [
                {"type": "text",
                 "text": (
                    task_guidance + "\n\n" + context_text +
                    f"\n\nQuestion: {query}\n"
                    "Answer in ~90–130 words. After each claim, cite the evidence id like [TEXT #2] or [FIGURE #1]. "
                    "Include: (1) how candidates are generated (top-k per MEDUSA head), "
                    "(2) who attends to whom (sparse ancestor-only mask), "
                    "(3) one concrete example from the context (levels/nodes).\nAnswer:"
                 )}
            ]
        )
    }]

    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    model_inputs = processor(text=prompt, images=images or None, return_tensors="pt").to(device)

    tok = processor.tokenizer
    eos_id = tok.eos_token_id
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else eos_id
    return model_inputs, eos_id, pad_id


# Generation Configuration — Two Decoding Strategies

In this section, we control **how the multimodal model generates answers** from the retrieved paper context and figures.  
You can choose between two decoding strategies by setting the `SELECT_OPTION` variable:

#### Option A — Deterministic, Factual (Beam Search)
- Uses **beam search** with multiple parallel beams to find the most likely output sequence.  
- Prioritizes **accuracy, stability, and reproducibility** — best for academic or factual QA tasks.  
- Every run produces the **same output** for a given prompt.  
- Recommended for: paper summarization, method explanation, and report-style responses.

#### Option B — Controlled Sampling (Diverse Wording)
- Uses **top-p sampling** with temperature control to introduce mild randomness.  
- Encourages **more natural phrasing and stylistic variation** while staying on topic.  
- Each run can produce slightly different, but semantically similar, answers.  
- Recommended for: creative descriptions, paraphrasing, or generating multiple reworded explanations.

Both options share the same safety constraints:
- `no_repeat_ngram_size=3` prevents repetitive phrasing.
- `eos_token_id` and `pad_token_id` ensure clean termination.
- The generated text is decoded **without reprinting the full prompt** (only the new tokens).

Use **Option A** for formal research outputs and **Option B** for exploratory or presentation-friendly answers.


In [50]:
# --- Cell 2: decoding config + run() ---
import re
SELECT_OPTION = "A"  # "A" deterministic, "B" controlled sampling

def get_gen_kwargs(eos_id, pad_id):
    if SELECT_OPTION.upper() == "A":
        return dict(
            max_new_tokens=480,
            min_new_tokens=120,
            num_beams=3,
            length_penalty=1.0,
            early_stopping=False,
            eos_token_id=eos_id, pad_token_id=pad_id,
            no_repeat_ngram_size=3,
        )
    else:
        return dict(
            max_new_tokens=480,
            min_new_tokens=120,
            do_sample=True, temperature=0.4, top_p=0.92,
            repetition_penalty=1.1,
            eos_token_id=eos_id, pad_token_id=pad_id,
            no_repeat_ngram_size=3,
        )

# Clean citation and strip URLs / numeric refs
def clean_answer(s: str) -> str:
    s = re.sub(r"\[TEXT\s*#\s*(\d+)\]", r"[TEXT #\1]", s, flags=re.I)
    s = re.sub(r"\[FIG(?:URE)?\s*#\s*(\d+)\]", r"[FIGURE #\1]", s, flags=re.I)
    s = re.sub(r"\[(\d{1,3})\]", "", s)             # drop [1], [23]
    s = re.sub(r"https?://\S+", "", s)              # remove URLs
    return s.strip()

def run_query(query: str):
    model_inputs, eos_id, pad_id = prepare_inputs(query)
    gen_kwargs = get_gen_kwargs(eos_id, pad_id)
    with torch.no_grad():
        output_ids = model.generate(**model_inputs, **gen_kwargs)

    full_text = processor.decode(output_ids[0], skip_special_tokens=True)
    sentinel = "Answer:"
    segment = full_text.split(sentinel)[-1] if sentinel in full_text else full_text
    answer = segment.removeprefix("ASSISTANT:").strip()

    answer = clean_answer(answer)
    print(answer)
    return answer


In [ ]:
# --- Cell 3: query execution ---
query = "Explain tree attention in Medusa paper?"
_ = run_query(query)

# Load LLaMA Model 

In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import is_flash_attn_2_available

# --- Model choice ---
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
use_quantization_config = True

# --- Quantization (for T4) ---
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# --- Flash Attention or SDPA ---
if is_flash_attn_2_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "sdpa"
print(f"[INFO] Using attention implementation: {attn_implementation}")

# --- Load tokenizer & model ---
start = time.time()
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=quantization_config if use_quantization_config else None,
    attn_implementation=attn_implementation,
    low_cpu_mem_usage=False,
    trust_remote_code=True
)

if not use_quantization_config:
    llm_model.to("cuda")

print(f"Model {model_id} loaded in {time.time()-start:.1f}s")


In [24]:
question = "Tell me about the 'LLM in a Flash' paper. What is the novelty? what methods they proposed and what inference speed up they got?"

# --- Build the RAG context from retrieved passages ---
context = build_context(question, k_text=10, k_image=5)

# --- Compose the final prompt ---
prompt = f"""
You are an expert research assistant specializing in AI and Machine Learning papers.
Your task is to answer questions clearly, factually, and concisely using only the provided retrieved context.

Guidelines:
- Base your answer only on the given context. Do not add external knowledge.
- If the context contains multiple relevant parts (text, tables, or figures), integrate them logically.
- If the context does not include the answer, say "Not mentioned in the retrieved context."
- Keep explanations precise and academic (1–2 short paragraphs maximum).

Question: {question}

Context:
{context}

Answer:
"""


In [25]:
# Tokenize the composed prompt
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate the model’s answer
with torch.no_grad():
    output_tokens = llm_model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and clean the output
answer = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

# Extract the part after "Answer:" if the model echoes the prompt
if "Answer:" in answer:
    answer = answer.split("Answer:")[-1].strip()

print("💬 RAG Answer:\n")
print(answer)


💬 RAG Answer:

The 'LLM in a Flash' paper proposes a method to speed up inference by selectively loading parameters on demand for each token generation step. The novelty of this approach is that it can achieve a 3x speed up over the naive baseline (Table 3) while performing better than the hybrid model, which is the theoretical lower bound for approaches that don't use sparsity. The method was applied to the Phi-2 model, and the inference speed-up was achieved by modifying the window size to ensure it never exceeds the limit. The results show that the approach can be effective even when the model is already small, as demonstrated by the application to the Phi-2 model.


In [17]:
question = "Tell me about the 'LLM in a Flash' paper. What is the novelty, what methods they proposed, and what inference speed up they got?"

text_hits, image_hits = retrieve_multimodal(question, k_text=5, k_image=3)

# Build text context from top hits
text_context = "\n\n".join([f"[Text: page {t['page']}] {t['content_preview']}" for t in text_hits])

# Collect image paths
image_paths = [h["image_path"] for h in image_hits]


In [18]:
prompt = f"""
You are an expert AI researcher summarizing the paper based only on the retrieved content.
Integrate textual and visual information logically.
If an image is included, describe what it shows and how it relates to the text.
Keep your answer academic and concise (1–2 paragraphs).

Question: {question}

Context:
{text_context}

Answer:
"""


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig

model_id = "microsoft/phi-3-vision-128k-instruct"

# 4-bit quantization for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load processor and model
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Phi-3-Vision loaded successfully on T4 in 4-bit mode.")
